In [ ]:
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict

def plot_calcium_traces_individual(input_folder, output_folder, skip_frames=[12, 23, 34, 45, 56, 67, 78, 89]):
    """
    Plot calcium imaging traces (raw mean gray value) for each sequence individually.
    No baseline correction or normalization - just raw intensity values.
    Groups are determined by all name elements EXCEPT the last TWO (which are C_##.tiff).
    Also exports an Excel file with all raw intensity values.
    Skips specific frame numbers and handles missing images gracefully.
    
    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_folder : str
        Path to save the plot images and Excel file.
    skip_frames : list
        List of frame indices to skip (default: [12, 23, 34, 45, 56, 67, 78, 89])
    
    Returns
    -------
    None
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Dictionary to group images by all elements except last two
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):
            # Remove file extension
            name_without_ext = os.path.splitext(file_name)[0]
            name_parts = name_without_ext.split("_")
            
            if len(name_parts) < 3:
                print(f"Warning: {file_name} doesn't have enough name elements. Skipping.")
                continue
            
            # Group by all elements EXCEPT last two (which should be C_## or similar)
            # e.g., "Co_100µMw3_15Glu_01_C_12.tiff" -> "Co_100µMw3_15Glu_01"
            # e.g., "Co_Ctrl_06_C_78.tiff" -> "Co_Ctrl_06"
            sequence_key = "_".join(name_parts[:-2])
            grouped_images[sequence_key].append(file_name)

    print(f"Found {len(grouped_images)} sequences")
    print(f"Skipping frames: {skip_frames}")
    
    # Dictionary to store all data for Excel export
    excel_data = {}
    
    # Process each SEQUENCE
    for sequence_key, file_names in sorted(grouped_images.items()):
        file_names.sort()
        
        print(f"\nProcessing sequence: {sequence_key} ({len(file_names)} images)")
        
        # Process images, skipping specified frames and handling missing images
        intensity_values = []
        frame_numbers = []
        skipped_frames_list = []
        missing_frames_list = []
        
        for frame_idx, file_name in enumerate(file_names):
            # Check if this frame should be skipped
            if frame_idx in skip_frames:
                skipped_frames_list.append(frame_idx)
                print(f"  Skipping frame {frame_idx} (in skip list)")
                continue
            
            img_path = os.path.join(input_folder, file_name)
            
            # Check if file exists
            if not os.path.exists(img_path):
                missing_frames_list.append(frame_idx)
                print(f"  Missing frame {frame_idx}: {file_name} not found")
                continue
            
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                missing_frames_list.append(frame_idx)
                print(f"  Warning: Unable to load frame {frame_idx}: {file_name}")
                continue
            
            # Calculate mean intensity of entire image
            mean_intensity = img.mean()
            intensity_values.append(mean_intensity)
            frame_numbers.append(frame_idx)
        
        if len(intensity_values) == 0:
            print(f"  Warning: No valid frames for {sequence_key}")
            continue
        
        print(f"  Processed {len(intensity_values)} frames")
        
        # Store data for Excel export
        excel_data[sequence_key] = {
            'Frame': frame_numbers,
            'Mean_Gray_Value': intensity_values
        }
        
        # Create individual plot for this sequence
        fig, ax = plt.subplots(figsize=(16, 7))
        
        # Plot the trace
        ax.plot(frame_numbers, intensity_values, marker='o', markersize=4, 
                color='#0068f9', linewidth=2)
        
        # Formatting
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Mean Gray Value', fontsize=12, fontweight='bold')
        ax.set_title(f'{sequence_key}\nRaw Mean Intensity per Frame', 
                     fontsize=13, fontweight='bold')
        
        # Set x-axis to show every 5th frame number for better legibility
        max_frame = len(file_names) - 1
        # Show every 5th tick, plus 0 and max_frame
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        
        # Also adjust y-axis tick label size for consistency
        ax.tick_params(axis='y', labelsize=9)
        
        ax.grid(True, alpha=0.3)
        
        # Add skipped/missing frames information at bottom
        info_lines = []
        if skipped_frames_list:
            info_lines.append("Skipped frames: " + ", ".join([str(frame) for frame in skipped_frames_list]))
        if missing_frames_list:
            info_lines.append("Missing frames: " + ", ".join([str(frame) for frame in missing_frames_list]))
        
        if info_lines:
            info_text = " | ".join(info_lines)
            fig.text(0.5, 0.02, info_text, ha='center', fontsize=8, style='italic', color='red')
            plt.subplots_adjust(bottom=0.12)
        else:
            plt.tight_layout()
        
        # Save with sequence name
        output_path = os.path.join(output_folder, f"{sequence_key}_Raw_Trace.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved plot to: {output_path}")
        if skipped_frames_list:
            print(f"  Skipped {len(skipped_frames_list)} frames: {skipped_frames_list}")
        if missing_frames_list:
            print(f"  Missing {len(missing_frames_list)} frames: {missing_frames_list}")
    
    # Export all data to Excel with multiple sheets
    if excel_data:
        excel_path = os.path.join(output_folder, "Raw_Intensity_Data.xlsx")
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for sequence_key, data in excel_data.items():
                # Convert to DataFrame
                df = pd.DataFrame(data)
                
                # Truncate sheet name to 31 characters (Excel limit)
                sheet_name = sequence_key[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"\n✓ Excel file saved to: {excel_path}")
        print(f"  Contains {len(excel_data)} sheets (one per sequence)")
    
    print(f"\nAll plots saved to: {output_folder}")


# Example usage
input_folder = '/Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All'
output_folder = os.path.join(input_folder, "0_Individual_Raw_Traces")

# Create raw intensity plots for each sequence
plot_calcium_traces_individual(input_folder, output_folder, skip_frames=[12, 23, 34, 45, 56, 67, 78, 89])

Found 81 sequences
Skipping frames: [12, 23, 34, 45, 56, 67, 78, 89]

Processing sequence: Co_100µMw3_15Glu_01 (90 images)
  Skipping frame 12 (in skip list)
  Skipping frame 23 (in skip list)
  Skipping frame 34 (in skip list)
  Skipping frame 45 (in skip list)
  Skipping frame 56 (in skip list)
  Skipping frame 67 (in skip list)
  Skipping frame 78 (in skip list)
  Skipping frame 89 (in skip list)
  AUC: 401.78
  Saved plot to: /Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces/Co_100µMw3_15Glu_01_Raw_Trace.png
  Skipped 8 frames: [12, 23, 34, 45, 56, 67, 78, 89]

Processing sequence: Co_100µMw3_15Glu_02 (90 images)
  Skipping frame 12 (in skip list)
  Skipping frame 23 (in skip list)
  Skipping frame 34 (in skip list)
  Skipping frame 45 (in skip list)
  Skipping frame 56 (in skip list)
  Skipping frame 67 (in skip list)
  Skipping frame 78 (in skip list)
  Skipping frame 89 (in skip list)
  AUC: 438.88
  Saved plot to: /Users/allisonpickle/Desktop/C

In [12]:
import matplotlib.pyplot as plt
import os
import pandas as pd
from collections import defaultdict

def plot_grouped_calcium_traces(input_excel, output_folder):
    """
    Create grouped plots of calcium imaging traces from Raw_Intensity_Data.xlsx.
    
    Creates two types of plots for both RAW and NORMALIZED data:
    1. Individual plots for each sequence (one sequence per plot)
    2. Group plots (all sequences from same group on one plot)
    
    Normalized data: F(t) / F₀ where F₀ is the first value in each sequence
    
    Groups are determined by all name elements EXCEPT the last element.
    E.g., "Co_100µMw3_15Glu_01" -> group "Co_100µMw3_15Glu"
          "Co_Ctrl_06" -> group "Co_Ctrl"
    
    Parameters
    ----------
    input_excel : str
        Path to the Raw_Intensity_Data.xlsx file
    output_folder : str
        Path to save the grouped plot images
    
    Returns
    -------
    None
    """
    # Create output folders for RAW data
    individual_folder_raw = os.path.join(output_folder, "Raw_Traces", "Individual_Traces")
    grouped_folder_raw = os.path.join(output_folder, "Raw_Traces", "Grouped_Traces")
    os.makedirs(individual_folder_raw, exist_ok=True)
    os.makedirs(grouped_folder_raw, exist_ok=True)
    
    # Create output folders for NORMALIZED data
    individual_folder_norm = os.path.join(output_folder, "Normalized_Traces", "Individual_Traces")
    grouped_folder_norm = os.path.join(output_folder, "Normalized_Traces", "Grouped_Traces")
    os.makedirs(individual_folder_norm, exist_ok=True)
    os.makedirs(grouped_folder_norm, exist_ok=True)
    
    # Read all sheets from Excel
    excel_file = pd.ExcelFile(input_excel)
    sheet_names = excel_file.sheet_names
    
    print(f"Found {len(sheet_names)} sequences in Excel file")
    print(f"Input file: {input_excel}\n")
    
    # Dictionary to group sequences
    grouped_sequences = defaultdict(list)
    
    # Read all data, normalize, and group sequences
    all_data_raw = {}
    all_data_norm = {}
    
    for sheet_name in sheet_names:
        df = pd.read_excel(input_excel, sheet_name=sheet_name)
        
        if 'Frame' not in df.columns or 'Mean_Gray_Value' not in df.columns:
            print(f"⚠️  Warning: {sheet_name} missing required columns. Skipping...")
            continue
        
        # Store raw data
        all_data_raw[sheet_name] = df.copy()
        
        # Calculate normalized data (F(t) / F₀)
        F0 = df['Mean_Gray_Value'].iloc[0]
        if F0 == 0:
            print(f"⚠️  Warning: {sheet_name} has F₀ = 0, skipping normalization...")
            continue
        
        df_norm = df.copy()
        df_norm['Normalized_Intensity'] = df['Mean_Gray_Value'] / F0
        all_data_norm[sheet_name] = df_norm
        
        # Group by all elements EXCEPT last element
        name_parts = sheet_name.split("_")
        if len(name_parts) >= 2:
            group_key = "_".join(name_parts[:-1])
        else:
            group_key = sheet_name
        
        grouped_sequences[group_key].append(sheet_name)
    
    print(f"Grouped into {len(grouped_sequences)} groups\n")
    
    # Color palette for grouped plots
    colors = ['#0068f9', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', 
              '#1abc9c', '#e67e22', '#3498db', '#95a5a6', '#c0392b']
    
    # ========== PART 1: RAW DATA - Individual Plots ==========
    print("="*80)
    print("CREATING RAW DATA - INDIVIDUAL PLOTS")
    print("="*80)
    
    for sequence_name, df in all_data_raw.items():
        print(f"  Plotting: {sequence_name}")
        
        fig, ax = plt.subplots(figsize=(16, 7))
        
        ax.plot(df['Frame'], df['Mean_Gray_Value'], marker='o', markersize=4,
                color='#0068f9', linewidth=2)
        
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Mean Gray Value', fontsize=12, fontweight='bold')
        ax.set_title(f'{sequence_name}\nRaw Mean Intensity per Frame',
                     fontsize=13, fontweight='bold')
        
        max_frame = int(df['Frame'].max())
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        output_path = os.path.join(individual_folder_raw, f"{sequence_name}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Saved {len(all_data_raw)} raw individual plots to: {individual_folder_raw}\n")
    
    # ========== PART 2: RAW DATA - Grouped Plots ==========
    print("="*80)
    print("CREATING RAW DATA - GROUPED PLOTS")
    print("="*80)
    
    for group_idx, (group_name, sequence_list) in enumerate(sorted(grouped_sequences.items())):
        print(f"\nGroup: {group_name} ({len(sequence_list)} sequences)")
        
        fig, ax = plt.subplots(figsize=(16, 7))
        
        for seq_idx, sequence_name in enumerate(sorted(sequence_list)):
            if sequence_name not in all_data_raw:
                continue
            
            df = all_data_raw[sequence_name]
            color = colors[seq_idx % len(colors)]
            
            ax.plot(df['Frame'], df['Mean_Gray_Value'], marker='o', markersize=4,
                    color=color, linewidth=2, alpha=0.8)
            
            print(f"  - {sequence_name}")
        
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Mean Gray Value', fontsize=12, fontweight='bold')
        ax.set_title(f'{group_name}\nRaw Mean Intensity per Frame (n={len(sequence_list)})',
                     fontsize=13, fontweight='bold')
        
        max_frame = 0
        for sequence_name in sequence_list:
            if sequence_name in all_data_raw:
                max_frame = max(max_frame, int(all_data_raw[sequence_name]['Frame'].max()))
        
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        output_path = os.path.join(grouped_folder_raw, f"{group_name}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Saved {len(grouped_sequences)} raw grouped plots to: {grouped_folder_raw}\n")
    
    # ========== PART 3: NORMALIZED DATA - Individual Plots ==========
    print("="*80)
    print("CREATING NORMALIZED DATA - INDIVIDUAL PLOTS")
    print("="*80)
    
    for sequence_name, df in all_data_norm.items():
        print(f"  Plotting: {sequence_name}")
        
        fig, ax = plt.subplots(figsize=(16, 7))
        
        ax.plot(df['Frame'], df['Normalized_Intensity'], marker='o', markersize=4,
                color='#2ecc71', linewidth=2)
        
        # Add horizontal line at y=1 (baseline)
        ax.axhline(y=1, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Normalized Intensity (F/F₀)', fontsize=12, fontweight='bold')
        ax.set_title(f'{sequence_name}\nNormalized Intensity per Frame',
                     fontsize=13, fontweight='bold')
        
        max_frame = int(df['Frame'].max())
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        output_path = os.path.join(individual_folder_norm, f"{sequence_name}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Saved {len(all_data_norm)} normalized individual plots to: {individual_folder_norm}\n")
    
    # ========== PART 4: NORMALIZED DATA - Grouped Plots ==========
    print("="*80)
    print("CREATING NORMALIZED DATA - GROUPED PLOTS")
    print("="*80)
    
    for group_idx, (group_name, sequence_list) in enumerate(sorted(grouped_sequences.items())):
        print(f"\nGroup: {group_name} ({len(sequence_list)} sequences)")
        
        fig, ax = plt.subplots(figsize=(16, 7))
        
        # Add horizontal line at y=1 (baseline)
        ax.axhline(y=1, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        for seq_idx, sequence_name in enumerate(sorted(sequence_list)):
            if sequence_name not in all_data_norm:
                continue
            
            df = all_data_norm[sequence_name]
            color = colors[seq_idx % len(colors)]
            
            ax.plot(df['Frame'], df['Normalized_Intensity'], marker='o', markersize=4,
                    color=color, linewidth=2, alpha=0.8)
            
            print(f"  - {sequence_name}")
        
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Normalized Intensity (F/F₀)', fontsize=12, fontweight='bold')
        ax.set_title(f'{group_name}\nNormalized Intensity per Frame (n={len(sequence_list)})',
                     fontsize=13, fontweight='bold')
        
        max_frame = 0
        for sequence_name in sequence_list:
            if sequence_name in all_data_norm:
                max_frame = max(max_frame, int(all_data_norm[sequence_name]['Frame'].max()))
        
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        output_path = os.path.join(grouped_folder_norm, f"{group_name}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Saved {len(grouped_sequences)} normalized grouped plots to: {grouped_folder_norm}\n")
    
    # ========== SUMMARY ==========
    print("="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total sequences plotted: {len(all_data_raw)}")
    print(f"Total groups: {len(grouped_sequences)}")
    print(f"\nRAW DATA:")
    print(f"  Individual plots: {individual_folder_raw}")
    print(f"  Grouped plots: {grouped_folder_raw}")
    print(f"\nNORMALIZED DATA:")
    print(f"  Individual plots: {individual_folder_norm}")
    print(f"  Grouped plots: {grouped_folder_norm}")
    print("="*80)


# Example usage
input_folder = '/Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces'
input_excel = os.path.join(input_folder, "Raw_Intensity_Data.xlsx")
output_folder = os.path.join(input_folder, "Plotted_Traces")

# Create grouped and individual plots (both raw and normalized)
plot_grouped_calcium_traces(input_excel, output_folder)

Found 79 sequences in Excel file
Input file: /Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces/Raw_Intensity_Data.xlsx

Grouped into 23 groups

CREATING RAW DATA - INDIVIDUAL PLOTS
  Plotting: Co_100µMw3_15Glu_01
  Plotting: Co_100µMw3_15Glu_02
  Plotting: Co_100µMw3_25Glu_01
  Plotting: Co_100µMw3_25Glu_02
  Plotting: Co_100µMw3_25Glu_03
  Plotting: Co_100µMw3_25Glu_04
  Plotting: Co_10µMw3_01
  Plotting: Co_10µMw3_02
  Plotting: Co_10µMw3_03
  Plotting: Co_10µMw3_15Glu_01
  Plotting: Co_10µMw3_15Glu_02
  Plotting: Co_10µMw3_15Glu_03
  Plotting: Co_10µMw3_25Glu_01
  Plotting: Co_10µMw3_25Glu_02
  Plotting: Co_10µMw3_25Glu_03
  Plotting: Co_10µMw3_25Glu_04
  Plotting: Co_10µMw3_25Glu_05
  Plotting: Co_10µMw3_25Glu_06
  Plotting: Co_10µMw3_5Glu_01
  Plotting: Co_10µMw3_5Glu_02
  Plotting: Co_15Glu_01
  Plotting: Co_15Glu_02
  Plotting: Co_15Glu_03
  Plotting: Co_20w3_5Glu_01
  Plotting: Co_20µMw3_25Glu_01
  Plotting: Co_20µMw3_25Glu_02
  Plotting: Co_20

In [8]:
import pandas as pd
import os
from scipy.integrate import trapezoid

def calculate_auc_from_excel(input_excel, output_excel=None):
    """
    Calculate AUC (Area Under Curve) for each sequence from Raw_Intensity_Data.xlsx.
    Reads each sheet and calculates AUC using the trapezoidal rule.
    
    Parameters
    ----------
    input_excel : str
        Path to the input Excel file (Raw_Intensity_Data.xlsx)
    output_excel : str, optional
        Path to save the AUC summary Excel file. If None, saves to same folder as input.
    
    Returns
    -------
    pd.DataFrame
        DataFrame containing AUC values for each sequence
    """
    
    # Read all sheets from the Excel file
    excel_file = pd.ExcelFile(input_excel)
    sheet_names = excel_file.sheet_names
    
    print(f"Found {len(sheet_names)} sequences in Excel file")
    print(f"Input file: {input_excel}\n")
    
    # List to store AUC data
    auc_data = []
    
    # Process each sheet
    for sheet_name in sheet_names:
        print(f"Processing: {sheet_name}")
        
        # Read the sheet
        df = pd.read_excel(input_excel, sheet_name=sheet_name)
        
        # Check if required columns exist
        if 'Frame' not in df.columns or 'Mean_Gray_Value' not in df.columns:
            print(f"  ⚠️  Warning: Missing required columns. Skipping...")
            continue
        
        # Remove any NaN values
        df_clean = df[['Frame', 'Mean_Gray_Value']].dropna()
        
        if len(df_clean) == 0:
            print(f"  ⚠️  Warning: No valid data. Skipping...")
            continue
        
        # Calculate AUC using trapezoidal rule
        auc = trapezoid(df_clean['Mean_Gray_Value'], df_clean['Frame'])
        
        print(f"  AUC: {auc:.2f} (calculated from {len(df_clean)} points)")
        
        # Store only sequence name and AUC
        auc_data.append({
            'Sequence': sheet_name,
            'AUC': auc,
            'Num_Frames': len(df_clean)
        })
    
    # Create DataFrame
    auc_df = pd.DataFrame(auc_data)
    
    # Determine output path
    if output_excel is None:
        input_dir = os.path.dirname(input_excel)
        output_excel = os.path.join(input_dir, "AUC_Summary.xlsx")
    
    # Save to Excel
    auc_df.to_excel(output_excel, index=False)
    
    print(f"\n{'='*80}")
    print(f"✓ AUC summary saved to: {output_excel}")
    print(f"  Contains AUC values for {len(auc_data)} sequences")
    print(f"{'='*80}\n")
    
    # Print summary statistics
    print("Summary Statistics:")
    print(f"  Average AUC: {auc_df['AUC'].mean():.2f}")
    print(f"  Median AUC: {auc_df['AUC'].median():.2f}")
    print(f"  Min AUC: {auc_df['AUC'].min():.2f} ({auc_df.loc[auc_df['AUC'].idxmin(), 'Sequence']})")
    print(f"  Max AUC: {auc_df['AUC'].max():.2f} ({auc_df.loc[auc_df['AUC'].idxmax(), 'Sequence']})")
    
    return auc_df


# Example usage
input_folder = '/Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces'
input_excel = os.path.join(input_folder, "Raw_Intensity_Data.xlsx")
output_excel = os.path.join(input_folder, "AUC_Summary.xlsx")

# Calculate AUC from existing Excel file
auc_results = calculate_auc_from_excel(input_excel, output_excel)

# Optional: Display the results
print("\nAUC Results:")
print(auc_results.to_string(index=False))

Found 79 sequences in Excel file
Input file: /Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces/Raw_Intensity_Data.xlsx

Processing: Co_100µMw3_15Glu_01
  AUC: 401.78 (calculated from 82 points)
Processing: Co_100µMw3_15Glu_02
  AUC: 438.88 (calculated from 82 points)
Processing: Co_100µMw3_25Glu_01
  AUC: 382.26 (calculated from 82 points)
Processing: Co_100µMw3_25Glu_02
  AUC: 393.81 (calculated from 82 points)
Processing: Co_100µMw3_25Glu_03
  AUC: 382.26 (calculated from 82 points)
Processing: Co_100µMw3_25Glu_04
  AUC: 393.81 (calculated from 82 points)
Processing: Co_10µMw3_01
  AUC: 234.16 (calculated from 82 points)
Processing: Co_10µMw3_02
  AUC: 436.22 (calculated from 75 points)
Processing: Co_10µMw3_03
  AUC: 269.49 (calculated from 82 points)
Processing: Co_10µMw3_15Glu_01
  AUC: 270.98 (calculated from 82 points)
Processing: Co_10µMw3_15Glu_02
  AUC: 399.77 (calculated from 82 points)
Processing: Co_10µMw3_15Glu_03
  AUC: 684.44 (calculate

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from collections import defaultdict

def calculate_delta_f_metrics(input_excel, output_excel, plot_output_folder):
    """
    Calculate ΔF/F₀ metrics for calcium imaging data.
    ΔF/F₀ = (F(t) - F₀) / F₀
    where F₀ is the first (baseline) value and F(t) is the intensity at each timepoint.
    
    Also calculates:
    - Peak ΔF/F₀ (maximum change)
    - Mean ΔF/F₀ (average change across all frames)
    - AUC of ΔF/F₀ curve
    
    Creates individual and grouped percentage plots.
    
    Parameters
    ----------
    input_excel : str
        Path to the input Excel file (Raw_Intensity_Data.xlsx)
    output_excel : str
        Path to save the ΔF/F₀ data Excel file
    plot_output_folder : str
        Path to save ΔF/F₀ trace plots (percentage format)
    
    Returns
    -------
    None
    """
    
    # Create output folders
    individual_folder = os.path.join(plot_output_folder, "Individual_Traces")
    grouped_folder = os.path.join(plot_output_folder, "Grouped_Traces")
    os.makedirs(individual_folder, exist_ok=True)
    os.makedirs(grouped_folder, exist_ok=True)
    
    # Read all sheets from the Excel file
    excel_file = pd.ExcelFile(input_excel)
    sheet_names = excel_file.sheet_names
    
    print(f"Found {len(sheet_names)} sheets to process")
    
    # Dictionary to store ΔF/F₀ data and group sequences
    delta_f_data = {}
    summary_data = []
    grouped_sequences = defaultdict(list)
    
    # Extended color palette for many sequences
    colors = ['#0068f9', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', 
              '#1abc9c', '#e67e22', '#3498db', '#95a5a6', '#c0392b',
              '#16a085', '#d35400', '#8e44ad', '#2980b9', '#c0392b',
              '#f1c40f', '#27ae60', '#e74c3c', '#3498db', '#9b59b6']
    
    # Process each sheet
    for sheet_name in sheet_names:
        print(f"\nProcessing sheet: {sheet_name}")
        
        # Read the sheet
        df = pd.read_excel(input_excel, sheet_name=sheet_name)
        
        # Check if Mean_Gray_Value column exists
        if 'Mean_Gray_Value' not in df.columns:
            print(f"  Warning: No 'Mean_Gray_Value' column found, skipping...")
            continue
        
        # Get F₀ (first non-NaN value as baseline)
        F0 = df['Mean_Gray_Value'].dropna().iloc[0]
        
        if F0 == 0:
            print(f"  Warning: F₀ is 0, cannot calculate ΔF/F₀. Skipping...")
            continue
        
        print(f"  F₀ (baseline): {F0:.2f}")
        
        # Calculate ΔF/F₀ = (F(t) - F₀) / F₀
        df['Delta_F_over_F0'] = (df['Mean_Gray_Value'] - F0) / F0
        
        # Calculate as percentage (multiply by 100)
        df['Delta_F_over_F0_Percent'] = df['Delta_F_over_F0'] * 100
        
        # Calculate metrics
        peak_delta_f = df['Delta_F_over_F0'].max()
        mean_delta_f = df['Delta_F_over_F0'].mean()
        auc_delta_f = trapezoid(df['Delta_F_over_F0'], df['Frame'])
        
        print(f"  Peak ΔF/F₀: {peak_delta_f:.4f} ({peak_delta_f*100:.2f}%)")
        print(f"  Mean ΔF/F₀: {mean_delta_f:.4f} ({mean_delta_f*100:.2f}%)")
        print(f"  AUC (ΔF/F₀): {auc_delta_f:.4f}")
        
        # Store summary data
        summary_data.append({
            'Sequence': sheet_name,
            'F0_Baseline': F0,
            'Peak_Delta_F_over_F0_Percent': peak_delta_f * 100,
            'Mean_Delta_F_over_F0_Percent': mean_delta_f * 100,
            'AUC_Delta_F_over_F0': auc_delta_f,
            'Num_Frames': len(df)
        })
        
        # Store ΔF/F₀ data
        delta_f_data[sheet_name] = df
        
        # Group by all elements EXCEPT last element
        name_parts = sheet_name.split("_")
        if len(name_parts) >= 2:
            group_key = "_".join(name_parts[:-1])
        else:
            group_key = sheet_name
        
        grouped_sequences[group_key].append(sheet_name)
        
        print(f"  ✓ Processed {len(df)} data points")
        
        # Determine x-axis tick positions
        max_frame = int(df['Frame'].max())
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        # ========== INDIVIDUAL PLOT ==========
        fig, ax = plt.subplots(figsize=(16, 7))
        
        # Plot the ΔF/F₀ trace in percentage
        ax.plot(df['Frame'], df['Delta_F_over_F0_Percent'], marker='o', markersize=4, 
                color='#9b59b6', linewidth=2)
        
        # Add horizontal line at y=0 (baseline)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        # Formatting
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('ΔF/F₀ (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{sheet_name}\nΔF/F₀ (%) per Frame (Peak: {peak_delta_f*100:.2f}%, Mean: {mean_delta_f*100:.2f}%, AUC: {auc_delta_f:.3f})', 
                     fontsize=13, fontweight='bold')
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save individual plot
        plot_path = os.path.join(individual_folder, f"{sheet_name}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  💾 Saved individual plot to: {os.path.basename(plot_path)}")
    
    # ========== GROUPED PLOTS ==========
    print("\n" + "="*80)
    print("CREATING GROUPED PLOTS")
    print("="*80)
    
    for group_name, sequence_list in sorted(grouped_sequences.items()):
        print(f"\nGroup: {group_name} ({len(sequence_list)} sequences)")
        
        fig, ax = plt.subplots(figsize=(16, 7))
        
        # Add horizontal line at y=0 (baseline)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        # Plot each sequence in the group
        for seq_idx, sequence_name in enumerate(sorted(sequence_list)):
            if sequence_name not in delta_f_data:
                continue
            
            df = delta_f_data[sequence_name]
            color = colors[seq_idx % len(colors)]
            
            ax.plot(df['Frame'], df['Delta_F_over_F0_Percent'], marker='o', markersize=4,
                    color=color, linewidth=2, alpha=0.8)
            
            print(f"  - {sequence_name}")
        
        # Formatting
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('ΔF/F₀ (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{group_name}\nΔF/F₀ (%) per Frame (n={len(sequence_list)})',
                     fontsize=13, fontweight='bold')
        
        # Get max frame across all sequences in group
        max_frame = 0
        for sequence_name in sequence_list:
            if sequence_name in delta_f_data:
                max_frame = max(max_frame, int(delta_f_data[sequence_name]['Frame'].max()))
        
        tick_positions = list(range(0, max_frame + 1, 5))
        if max_frame not in tick_positions:
            tick_positions.append(max_frame)
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_positions, fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        output_path = os.path.join(grouped_folder, f"{group_name}.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  💾 Saved grouped plot to: {os.path.basename(output_path)}")
    
    # Export ΔF/F₀ data to Excel
    if delta_f_data:
        with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
            for sheet_name, df in delta_f_data.items():
                # Truncate sheet name to 31 characters (Excel limit)
                sheet_name_truncated = sheet_name[:31]
                df.to_excel(writer, sheet_name=sheet_name_truncated, index=False)
        
        print(f"\n{'='*80}")
        print(f"✓ ΔF/F₀ data saved to: {output_excel}")
        print(f"  Contains {len(delta_f_data)} sheets")
        print(f"✓ Individual plots saved to: {individual_folder}")
        print(f"  Created {len(delta_f_data)} plots")
        print(f"✓ Grouped plots saved to: {grouped_folder}")
        print(f"  Created {len(grouped_sequences)} plots")
        print(f"{'='*80}")
    else:
        print("\n⚠️  No data to process!")
    
    # Export summary metrics to separate Excel file
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_excel_path = os.path.join(plot_output_folder, "Delta_F_over_F0_Summary.xlsx")
        summary_df.to_excel(summary_excel_path, index=False)
        
        print(f"\n✓ ΔF/F₀ summary metrics saved to: {summary_excel_path}")
        print(f"  Contains metrics for {len(summary_data)} sequences")


# Example usage
input_folder = '/Users/allisonpickle/Desktop/Calcium_Imaging_All_Plates/All/0_Individual_Raw_Traces'
input_excel = os.path.join(input_folder, "Raw_Intensity_Data.xlsx")
output_excel = os.path.join(input_folder, "∆Fluorescence over First Image Percentage.xlsx")
plot_output_folder = os.path.join(input_folder, "∆Fluorescence over First Image Percentage")

# Calculate ΔF/F₀ metrics and create plots
calculate_delta_f_metrics(input_excel, output_excel, plot_output_folder)

Found 79 sheets to process

Processing sheet: Co_100µMw3_15Glu_01
  F₀ (baseline): 1.52
  Peak ΔF/F₀: 2.5234 (252.34%)
  Mean ΔF/F₀: 1.9903 (199.03%)
  AUC (ΔF/F₀): 176.4124
  ✓ Processed 82 data points
  💾 Saved individual plot to: Co_100µMw3_15Glu_01.png

Processing sheet: Co_100µMw3_15Glu_02
  F₀ (baseline): 2.15
  Peak ΔF/F₀: 1.7356 (173.56%)
  Mean ΔF/F₀: 1.3159 (131.59%)
  AUC (ΔF/F₀): 116.1742
  ✓ Processed 82 data points
  💾 Saved individual plot to: Co_100µMw3_15Glu_02.png

Processing sheet: Co_100µMw3_25Glu_01
  F₀ (baseline): 1.99
  Peak ΔF/F₀: 1.4147 (141.47%)
  Mean ΔF/F₀: 1.1714 (117.14%)
  AUC (ΔF/F₀): 103.7510
  ✓ Processed 82 data points
  💾 Saved individual plot to: Co_100µMw3_25Glu_01.png

Processing sheet: Co_100µMw3_25Glu_02
  F₀ (baseline): 1.81
  Peak ΔF/F₀: 1.7793 (177.93%)
  Mean ΔF/F₀: 1.4626 (146.26%)
  AUC (ΔF/F₀): 130.0165
  ✓ Processed 82 data points
  💾 Saved individual plot to: Co_100µMw3_25Glu_02.png

Processing sheet: Co_100µMw3_25Glu_03
  F₀ (baseline